In [1]:
import os
from dotenv import load_dotenv
import huggingface_hub

load_dotenv()
huggingface_hub.login(token=os.getenv("HF_TOKEN"))

c:\Users\vkamat01\hedging-txtclf-experiments\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
import pandas as pd

In [3]:
from datasets import load_dataset

In [4]:
# Load the dataset
ds = load_dataset("hfmlsoc/sp500_dataset_earnings_calls")

In [ ]:
rec = ds["train"][0]
print(rec["2023_earnings_q1"])

In [6]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['Symbol', 'Security', 'GICS Sector', 'GICS Sub-Industry', 'CIK', '2022_earnings_q1', '2022_earnings_q2', '2022_earnings_q3', '2022_earnings_q4', '2025_earnings_q1', '2023_earnings_q1', '2023_earnings_q2', '2023_earnings_q3', '2023_earnings_q4', '2024_earnings_q1', '2024_earnings_q2', '2024_earnings_q3', '2024_earnings_q4'],
        num_rows: 503
    })
})


In [ ]:
rec = ds["train"][0]
for key in rec.keys():
    val = str(rec[key])
    print(f"{key}: {val[:150]}")

In [8]:
# How many companies?
print(len(ds["train"]))

# How long is a typical transcript?
rec = ds["train"][0]
print(len(rec["2023_earnings_q1"]))

503
60087


In [9]:
print(ds["train"][0]["2023_earnings_q1"][:2000])

Operator: Ladies and gentlemen, thank you for standing by. Welcome to the 3M First Quarter Earnings Conference Call. During the presentation, all participants will be in a listen-only mode. Afterwards, we will conduct a question-and-answer session. [Operator Instructions] As a reminder, this conference is being recorded, Tuesday, April 25, 2023. I would now like to turn the call over to Bruce Jermeland, Senior Vice President of Investor Relations at 3M.
Bruce Jermeland: Thank you, and good morning, everyone, and welcome to our fourth quarter earnings conference call. With me today are Mike Roman, 3M’s Chairman and Chief Executive Officer and Monish Patolawala, our Chief Financial and Transformation Officer. Mike and Monish will make some formal comments then we will take your questions. Please note that today’s earnings release and slide presentation accompanying this call are posted on the homepage of our Investor Relations website at 3m.com. Please turn to Slide 2. Please take a mome

# Speaker Distribution
Parse speaker turns from transcripts to understand the mix of executives, analysts, and operators across companies.

In [ ]:
from collections import Counter

def get_speakers(transcript_text):
    speakers = []
    for line in transcript_text.split("\n"):
        if ":" in line:
            speaker = line.split(":")[0].strip()
            if len(speaker) < 40:  # filter out non-speaker lines
                speakers.append(speaker)
    return speakers

# Check speaker distribution across 5 companies
for i in range(5):
    rec = ds["train"][i]
    speakers = get_speakers(rec["2023_earnings_q1"] or "")
    print(f"\n{rec['Symbol']} ({rec['GICS Sector']}):")
    print(Counter(speakers).most_common(5))

# Transcript Scale
Measure transcript length and speaker turn counts to assess the volume of extractable sentence-level data.

In [11]:
import numpy as np

lengths = []
turn_counts = []

for rec in ds["train"]:
    text = rec["2023_earnings_q1"]
    if text:
        lengths.append(len(text))
        turns = [l for l in text.split("\n") if ":" in l and len(l.split(":")[0]) < 40]
        turn_counts.append(len(turns))

print(f"Transcript length — median: {np.median(lengths):.0f}, min: {min(lengths)}, max: {max(lengths)}")
print(f"Speaker turns — median: {np.median(turn_counts):.0f}, min: {min(turn_counts)}, max: {max(turn_counts)}")

Transcript length — median: 51893, min: 12169, max: 226464
Speaker turns — median: 66, min: 5, max: 183


Check how many quarterly columns have missing/null transcripts across the dataset:


In [12]:
quarters = ["2022_earnings_q1", "2022_earnings_q2", "2022_earnings_q3", "2022_earnings_q4",
            "2023_earnings_q1", "2023_earnings_q2", "2023_earnings_q3", "2023_earnings_q4",
            "2024_earnings_q1", "2024_earnings_q2", "2024_earnings_q3", "2024_earnings_q4",
            "2025_earnings_q1"]

for q in quarters:
    missing = sum(1 for rec in ds["train"] if not rec[q])
    print(f"{q}: {missing} missing out of {len(ds['train'])}")

2022_earnings_q1: 22 missing out of 503
2022_earnings_q2: 21 missing out of 503
2022_earnings_q3: 21 missing out of 503
2022_earnings_q4: 20 missing out of 503
2023_earnings_q1: 18 missing out of 503
2023_earnings_q2: 21 missing out of 503
2023_earnings_q3: 14 missing out of 503
2023_earnings_q4: 14 missing out of 503
2024_earnings_q1: 13 missing out of 503
2024_earnings_q2: 13 missing out of 503
2024_earnings_q3: 12 missing out of 503
2024_earnings_q4: 16 missing out of 503
2025_earnings_q1: 25 missing out of 503


## Reshape to Long Format
Convert wide format (one row per company, one column per quarter) to long format 
(one row per transcript). Filter out missing and very short transcripts (< 5000 chars).

In [13]:
import pandas as pd

quarters = ["2022_earnings_q1", "2022_earnings_q2", "2022_earnings_q3", "2022_earnings_q4",
            "2023_earnings_q1", "2023_earnings_q2", "2023_earnings_q3", "2023_earnings_q4",
            "2024_earnings_q1", "2024_earnings_q2", "2024_earnings_q3", "2024_earnings_q4",
            "2025_earnings_q1"]

rows = []
for rec in ds["train"]:
    for q in quarters:
        text = rec[q]
        if text and len(text) > 5000:  # filter very short/incomplete
            rows.append({
                "symbol": rec["Symbol"],
                "sector": rec["GICS Sector"],
                "quarter": q,
                "text": text
            })

df = pd.DataFrame(rows)
print(f"Total transcripts: {len(df)}")
print(df["sector"].value_counts())

Total transcripts: 6309
sector
Industrials               985
Financials                914
Information Technology    869
Health Care               768
Consumer Discretionary    640
Consumer Staples          476
Real Estate               402
Utilities                 387
Materials                 331
Energy                    271
Communication Services    266
Name: count, dtype: int64


## Dataset Scale and Sector Distribution
6,309 transcripts available after filtering. Good sector diversity across all 11 GICS 
sectors — useful for ensuring synthetic data generation isn't sector-biased.

# Saving this reshaped dataframe to data/raw 

In [17]:
import os
os.chdir("..")  # move up to project root
df.to_parquet("data/raw/transcripts_long.parquet", index=False)
print("Saved.")

Saved.


In [18]:
## Sentence Tokenization Test
## Split one transcript into sentences to verify tokenization quality 
## before applying to the full dataset.

import nltk
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize

sample = df.iloc[0]["text"]
sentences = sent_tokenize(sample)
print(f"Total sentences: {len(sentences)}")
for s in sentences[:10]:
    print(s)
    print("---")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\vkamat01\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


Total sentences: 662
Operator: Ladies and gentlemen, thank you for standing by.
---
Welcome to the 3M First Quarter Earnings Conference Call.
---
During the presentation, all participants will be in a listen-only mode.
---
Afterwards, we will conduct a question-and-answer session.
---
[Operator Instructions] As a reminder, this conference is being recorded, Tuesday, April 26, 2022.
---
I would now like to turn the call over to Bruce Jermeland, Senior Vice President of Investor Relations at 3M.
---
Bruce Jermeland: Thank you and good morning, everyone and welcome to our first quarter earnings conference call.
---
With me today are Mike Roman, 3M’s Chairman and Chief Executive Officer; Monish Patolawala, our Chief Financial and Transformation Officer; and John Banovetz, our Chief Technology Officer.
---
John is joining us today to discuss our progress on the sustainability goals that we introduced in February last year.
---
Mike, Monash and John will make some formal comments, then we wi

In [27]:
## Sentence Cleaning Function
## Filters out operator lines, boilerplate, and strips speaker name prefixes
## before sentence-level analysis.
BOILERPLATE_PHRASES = [
    "ladies and gentlemen",
    "thank you for standing by",
    "listen-only mode",
    "question-and-answer session",
    "conference is being recorded",
    "turn the call over",
    "investor relations website",
    "forward-looking statement",
    "non-gaap",
    "reconciliation",
    "slide presentation",
    "earnings release",
    "welcome to the",
    "good morning",
    "good afternoon", 
    "turn to slide",
    "form 10-k",
    "risk factors",
    "please note",
    "make some formal comments",
    "take your questions",
    "with me today",
    "i would like to take a moment",
    "that concludes",
    "turn the call back",
    "thank you for joining",
    "please disconnect",
    "question-and-answer portion",
]

def is_boilerplate(s):
    s_lower = s.lower()
    return any(phrase in s_lower for phrase in BOILERPLATE_PHRASES)

def clean_sentences(sentences):
    cleaned = []
    for s in sentences:
        # Remove speaker prefix if present
        if ":" in s:
            prefix = s.split(":")[0].strip()
            if len(prefix) < 40 and " " in prefix or prefix == "Operator":
                s = s.split(":", 1)[1].strip()
        # Skip operator boilerplate and bracketed instructions
        if s.startswith("["):
            continue
        if len(s) < 20:
            continue
        if is_boilerplate(s):
            continue
        cleaned.append(s)
    return cleaned

# Test on sample
cleaned = clean_sentences(sentences)
print(f"Before: {len(sentences)} → After: {len(cleaned)}")
for s in cleaned[:10]:
    print(s)
    print("---")

Before: 662 → After: 527
John is joining us today to discuss our progress on the sustainability goals that we introduced in February last year.
---
During today's conference call, we will be making certain predictive statements that reflect our current views about 3M's future performance and financial results.
---
These statements are based on certain assumptions and expectations of future events that are subject to risks and uncertainties.
---
We recognize that the increases in legal-related charges that we have incurred the past couple of years have impacted investors' understanding of our underlying financial and operating performance.
---
This change is a result of discussions we have had with many of you, along with recent benchmarking work we have done.
---
Also, our Q1 2022 financial performance and full year 2022 guidance in today's press release and presentation incorporate these changes.
---
As you can see, operating margins in 2021 were 22.2% on this new adjusted basis, or u

In [28]:
for s in cleaned[-10:]:
    print(s)
    print("---")

And then there's ongoing operations.
---
And so we're working on both of those in Zwijndrecht as the charge that we announced in March was to resolve remediation related to that historical PFOA-PFOS manufacturing.
---
And then we're working with the authorities and Flanders around an operating permit going forward.
---
And that's something that we've been doing around the world at our five sites with regulatory authorities and continue to do that.
---
It's – PFAS continues to be a critical PFAS substances.
---
There's more than 4,000 of them.
---
We continue to have some PFAS in our products that is critical to customer needs in health care, electronics, automotive.
---
And so the -- its something we’re managing with those sites, managing with those authorities and something we'll keep you updated on as appropriate as we go forward.
---
To wrap up, we had a good start to the year with solid growth, sequential margin expansion and strong cash generation.
---
We are positioned for a succ

## Sentence Extraction and Dual-Version Save

This cell applies sentence tokenization and cleaning to the full dataset (6,309 transcripts)
and produces two saved outputs that serve different purposes downstream.

### Why two versions?
- `sentences_clean.parquet` — boilerplate removed, genuine executive speech only.
  Used for: positive hedging example mining, true negative mining, and LLM prompting.
- `sentences_all.parquet` — minimal filtering only, includes operator lines and boilerplate.
  Used for: mass negative construction, where unambiguously non-hedging text is useful
  for creating class imbalance that mirrors real-world conditions.

### What the cleaning function does
- Strips speaker name prefixes (e.g. "Mike Roman: ...")
- Removes bracketed instructions (e.g. [Operator Instructions])
- Filters sentences shorter than 20 characters
- Removes boilerplate phrases (greetings, call logistics, slide references, closing remarks)

### Output schema
Both parquet files share the same columns:
- `symbol` — company ticker
- `sector` — GICS sector
- `quarter` — earnings quarter (e.g. 2023_earnings_q1)
- `sentence` — individual sentence text

### Design note
The cleaning logic is currently embedded in this notebook for exploration purposes.
It will be migrated to `src/data/preprocess.py` once finalized.

In [29]:
# Apply cleaning to full dataset — save two versions
all_records = []

for _, row in df.iterrows():
    sentences_raw = sent_tokenize(row["text"])
    sentences_cleaned = clean_sentences(sentences_raw)
    
    for s in sentences_raw:
        all_records.append({
            "symbol": row["symbol"],
            "sector": row["sector"],
            "quarter": row["quarter"],
            "sentence": s,
            "cleaned": True if s in set(sentences_cleaned) else False
        })

df_sentences = pd.DataFrame(all_records)
df_clean = df_sentences[df_sentences["cleaned"] == True].drop(columns="cleaned")
df_all = df_sentences.drop(columns="cleaned")

print(f"All sentences: {len(df_all)}")
print(f"Cleaned sentences: {len(df_clean)}")

df_clean.to_parquet("data/raw/sentences_clean.parquet", index=False)
df_all.to_parquet("data/raw/sentences_all.parquet", index=False)
print("Saved.")

All sentences: 3171929
Cleaned sentences: 2505229
Saved.
